# Multimodal - Data Preparation and Transformation with Restrictions

This file is used to prepare and transform the groceries data. The following restrictions apply to this file:

## Restrictions:

1. Baskets with less than 30 items
2. Customers with less than 5 baskets

## Main Process and Steps:

### 1. Parameter Setup and Data Overview:
Set up the packages, path, and dataset name. It also includes an overview of the data, such as the date period of the data.

### 2. Indexing of Items:
Create indices to represent the items and a mapping table for reference.

### 3. Indexing of Customers:
Create indices to represent the customer numbers and a mapping table for reference.

### 4. Baskets for Each Customer and Items in Each Basket:
By utilizing timestamps, it is possible to determine which items were purchased together, group them into baskets, and identify the number of baskets each customer has.

### 5. Apply Restrictions:
Apply the specified restrictions to the data.

### 6. Separate Train and Test Datasets:
Split the dataset so that the number of baskets in the training set is equal to the number of baskets in the testing set. If a customer has an odd number of baskets, delete the latest date (last row).

### 7. Generate 3 Files as Input for the Model:
Generate the following files: `train_u2b.txt`, `train_b2i.txt`, `test_b2i.txt` and `test_u2b.txt`.

## Coding

### 1. Parameter Setup and Data Overview:

In [1]:
# Import packages
import pandas as pd
import numpy as np

In [2]:
# Set Path
path_name = r'D:\Jupyter\4742\Summer2024\FinalSubmission\Code\DataPreparationandTransformation\multimodal_data\data\multimodalwithres'

In [3]:
# Set and read dataset
df = pd.read_csv(path_name + '/Multimodal.csv')
df.head(5)

,LOC_ID,CUSTOMER_ID,TX_ID,TX_DATE,TX_TME,ITEM_ID,SUBGROUP_ID,NET_SALES_UNITS,NET_SALES_AMT
0,8,39260,748538,3/3/19,153300,6100,459,0.949816,3.854712
1,8,30743,876237,3/3/19,201000,1746,974,1.180286,1.834784
2,8,30743,876237,3/3/19,201000,1746,974,1.092798,1.699623
3,8,28346,746752,3/3/19,161400,1315,795,1.039463,2.955283
4,8,77899,925250,3/3/19,154100,8854,699,0.862407,3.591753


In [4]:
# The date period of the data
# Convert 'TX_DATE' column to datetime format
def parse_date(date_str):
    for fmt in ('%m/%d/%Y', '%m/%d/%y'):  # List formats to try
        try:
            return pd.to_datetime(date_str, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df['TX_DATE'] = df['TX_DATE'].apply(parse_date)

# Find first and last dates of data
first_date = df['TX_DATE'].min()
last_date = df['TX_DATE'].max()

print(f"First date in the dataset: {first_date}")
print(f"Last date in the dataset: {last_date}")

First date in the dataset: 2019-03-03 00:00:00
Last date in the dataset: 2020-02-21 00:00:00


### 2. Indexing of Items:

In [5]:
# Factorize the ITEM_ID column
df['item_no'], item_labels = pd.factorize(df['ITEM_ID'])

# Create a mapping table
item_mapping = pd.DataFrame({
    'ITEM_ID': item_labels,
    'item_no': range(len(item_labels))
})

# Drop the original ITEM_ID column
df = df.drop(columns=['LOC_ID', 'TX_ID', 'TX_TME', 'ITEM_ID', 'SUBGROUP_ID', 'NET_SALES_UNITS', 'NET_SALES_AMT'])
df

,CUSTOMER_ID,TX_DATE,item_no
0,39260,2019-03-03,0
1,30743,2019-03-03,1
2,30743,2019-03-03,1
3,28346,2019-03-03,2
4,77899,2019-03-03,3
...,...,...,...
790422,88580,2020-02-20,807
790423,20238,2020-02-20,113
790424,20238,2020-02-20,113
790425,20238,2020-02-20,916


### 3. Indexing of Customers:

In [6]:
# Factorize the CUSTOMER_ID column
df['uid'], item_labels = pd.factorize(df['CUSTOMER_ID'])

# Create a mapping table
item_mapping = pd.DataFrame({
    'CUSTOMER_ID': item_labels,
    'uid': range(len(item_labels))
})

# Drop the original CUSTOMER_ID column
df = df.drop(columns=['CUSTOMER_ID'])
df

,TX_DATE,item_no,uid
0,2019-03-03,0,0
1,2019-03-03,1,1
2,2019-03-03,1,1
3,2019-03-03,2,2
4,2019-03-03,3,3
...,...,...,...
790422,2020-02-20,807,10953
790423,2020-02-20,113,6477
790424,2020-02-20,113,6477
790425,2020-02-20,916,6477


### 4. Baskets for Each Customer and Items in Each Basket:

In [20]:
# Convert the 'TX_DATE' column to datetime format
df['TX_DATE'] = pd.to_datetime(df['TX_DATE'], format='%d-%m-%Y')

# Group data by 'uid' and 'Date' to create baskets for each customer and items in each basket
grouped_df = df.groupby(['uid', 'TX_DATE'])['item_no'].apply(list).reset_index()
expanded_df = grouped_df['item_no'].apply(pd.Series).rename(columns=lambda x: str(x+1))
expanded_df = expanded_df.fillna('')
for col in expanded_df.columns[:]:
    expanded_df[col] = expanded_df[col].apply(lambda x: str(int(x)) if x != '' else x)

final_df = pd.concat([grouped_df[['uid', 'TX_DATE']], expanded_df], axis=1)
final_df.rename(columns={'TX_DATE': 'TX_DATE'}, inplace=True)

final_df

,uid,TX_DATE,1,2,3,4,5,6,7,8,...,516,517,518,519,520,521,522,523,524,525
0,0,2019-03-03,0,24,24,26,27,28,46,47,...,,,,,,,,,,
1,0,2019-03-24,28,28,583,583,99,99,24,24,...,,,,,,,,,,
2,0,2019-04-07,28,24,587,2420,,,,,...,,,,,,,,,,
3,0,2019-04-14,152,1163,1163,1163,397,397,152,152,...,,,,,,,,,,
4,0,2019-04-24,35,35,35,328,955,955,955,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239255,37346,2020-02-20,693,,,,,,,,...,,,,,,,,,,
239256,37347,2020-02-20,410,410,410,410,,,,,...,,,,,,,,,,
239257,37348,2020-02-21,3888,,,,,,,,...,,,,,,,,,,
239258,37349,2020-02-21,433,731,731,731,151,1325,1325,1325,...,,,,,,,,,,


In [21]:
print("Columns in final_df after concatenation:", final_df.columns)

Columns in final_df after concatenation: Index(['uid', 'TX_DATE', '1', '2', '3', '4', '5', '6', '7', '8',
       ...
       '516', '517', '518', '519', '520', '521', '522', '523', '524', '525'],
      dtype='object', length=527)


### 5. Apply Restrictions:

In [22]:
# Baskets with less than 30 items
# Define a function to count items
def count_valid_values(row):
    return row.replace('', pd.NA).dropna().astype(bool).sum()

final_df = final_df.assign(valid_count=final_df.apply(count_valid_values, axis=1))\
                    .query('valid_count <= 31')

# Trim columns to the maximum number of valid values
max_valid_count = final_df['valid_count'].max()
final_df = final_df.drop(columns=['valid_count'])

def trim_columns(row, max_count):
    valid_values = row[row != '']
    if len(valid_values) > max_count:
        trimmed = valid_values.iloc[:max_count].tolist()
    else:
        trimmed = valid_values.tolist()
    return trimmed + [''] * (max_count - len(trimmed))

trimmed_data = final_df.apply(lambda row: trim_columns(row, max_valid_count), axis=1)
df_trimmed = pd.DataFrame(trimmed_data.tolist(), index=final_df.index)
final_df = df_trimmed.rename(columns={df_trimmed.columns[0]: 'uid'})
final_df

,uid,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
0,0,2019-03-03,0,24,24,26,27,28,46,47,...,,,,,,,,,,
1,0,2019-03-24,28,28,583,583,99,99,24,24,...,,,,,,,,,,
2,0,2019-04-07,28,24,587,2420,,,,,...,,,,,,,,,,
3,0,2019-04-14,152,1163,1163,1163,397,397,152,152,...,,,,,,,,,,
4,0,2019-04-24,35,35,35,328,955,955,955,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239255,37346,2020-02-20,693,,,,,,,,...,,,,,,,,,,
239256,37347,2020-02-20,410,410,410,410,,,,,...,,,,,,,,,,
239257,37348,2020-02-21,3888,,,,,,,,...,,,,,,,,,,
239258,37349,2020-02-21,433,731,731,731,151,1325,1325,1325,...,,,,,,,,,,


In [23]:
# Customers with less than 5 baskets
# Keep the first 10 rows
final_df = final_df.groupby('uid').apply(lambda x: x.head(10)).reset_index(drop=True)
final_df

C:\Users\41300\AppData\Local\Temp\ipykernel_9256\3845527340.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_df = final_df.groupby('uid').apply(lambda x: x.head(10)).reset_index(drop=True)


,uid,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
0,0,2019-03-03,0,24,24,26,27,28,46,47,...,,,,,,,,,,
1,0,2019-03-24,28,28,583,583,99,99,24,24,...,,,,,,,,,,
2,0,2019-04-07,28,24,587,2420,,,,,...,,,,,,,,,,
3,0,2019-04-14,152,1163,1163,1163,397,397,152,152,...,,,,,,,,,,
4,0,2019-04-24,35,35,35,328,955,955,955,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145462,37346,2020-02-20,693,,,,,,,,...,,,,,,,,,,
145463,37347,2020-02-20,410,410,410,410,,,,,...,,,,,,,,,,
145464,37348,2020-02-21,3888,,,,,,,,...,,,,,,,,,,
145465,37349,2020-02-21,433,731,731,731,151,1325,1325,1325,...,,,,,,,,,,


### 6. Separate Train and Test Datasets:

In [24]:
# Check data (before) - Count the number of basket for each customer
row_counts = final_df.groupby('uid').size().reset_index(name='counts')
row_counts

,uid,counts
0,0,10
1,1,10
2,2,10
3,3,10
4,4,10
...,...,...
37334,37346,1
37335,37347,1
37336,37348,1
37337,37349,1


In [25]:
# Count the number of basket for each customer
row_counts = final_df.groupby('uid').size().reset_index(name='counts')

# Identify customer with an odd number of basket
odd_members = row_counts[row_counts['counts'] % 2 == 1]['uid']

# For each identified customer, remove the last row (last date) to make the count even
def remove_last_row_if_odd(group):
    if len(group) % 2 == 1:
        return group[:-1]
    else:
        return group

final_df = final_df.loc[:, ~final_df.columns.duplicated()]

# filtered_df = final_df.groupby('uid', as_index=False).apply(remove_last_row_if_odd).reset_index(drop=True)
filtered_df = filtered_df.loc[:, ~filtered_df.columns.duplicated()]

filtered_df

,uid,TX_DATE,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
0,0,2019-03-03,0,24,24,26,27,28,46,47,...,,,,,,,,,,
1,0,2019-03-24,28,28,583,583,99,99,24,24,...,,,,,,,,,,
2,0,2019-04-07,28,24,587,2420,,,,,...,,,,,,,,,,
3,0,2019-04-14,152,1163,1163,1163,397,397,152,152,...,,,,,,,,,,
4,0,2019-04-24,35,35,35,328,955,955,955,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124413,37129,2020-02-20,17,,,,,,,,...,,,,,,,,,,
124414,37135,2020-02-17,2012,,,,,,,,...,,,,,,,,,,
124415,37135,2020-02-19,2467,,,,,,,,...,,,,,,,,,,
124416,37219,2020-02-18,899,,,,,,,,...,,,,,,,,,,


In [26]:
# Check data (after) - Count the number of basket for each customer
counts = filtered_df.groupby('uid').size().reset_index(name='counts')
counts

,uid,counts
0,0,10
1,1,10
2,2,10
3,3,10
4,4,10
...,...,...
22396,37098,2
22397,37101,2
22398,37129,4
22399,37135,2


In [27]:
# Split the data into training and test datasets
def split_data(group):
    half_size = len(group) // 2
    train_data = group.iloc[:half_size]
    test_data = group.iloc[half_size:]
    return train_data, test_data

# Split the data for each customer
train_list = []
test_list = []

for i, group in filtered_df.groupby('uid'):
    train_data, test_data = split_data(group)
    train_list.append(train_data)
    test_list.append(test_data)


train_df = pd.concat(train_list).reset_index(drop=True)
test_df = pd.concat(test_list).reset_index(drop=True)

train_df['TX_DATE'] = filtered_df['TX_DATE'].iloc[train_df.index].values
test_df['TX_DATE'] = filtered_df['TX_DATE'].iloc[test_df.index].values

train_df = train_df.loc[:, ~train_df.columns.duplicated()]
test_df = test_df.loc[:, ~test_df.columns.duplicated()]

print("Training Data:")
print(train_df)
print("\nTesting Data:")
print(test_df)

Training Data:
         uid    TX_DATE     2     3     4     5    6    7    8    9  ... 21  \
0          0 2019-03-03     0    24    24    26   27   28   46   47  ...      
1          0 2019-03-24    28    28   583   583   99   99   24   24  ...      
2          0 2019-04-07    28    24   587  2420                      ...      
3          0 2019-04-14   152  1163  1163  1163  397  397  152  152  ...      
4          0 2019-04-24    35    35    35   328  955  955  955       ...      
...      ...        ...   ...   ...   ...   ...  ...  ...  ...  ...  ... ..   
62204  37101 2020-01-17  2136                                        ...      
62205  37129 2020-02-20    17   100                                  ...      
62206  37129 2019-03-29    17  3543                                  ...      
62207  37135 2019-04-19  2012                                        ...      
62208  37219 2019-10-01   899                                        ...      

      22 23 24 25 26 27 28 29 30  
0

In [28]:
# Check data - Training set
# Count the number of customer
unique_members = train_df['uid'].nunique()
print("Number of unique Member_number entries:", unique_members)

# Count the number of basket for each customer
train_counts = train_df.groupby('uid').size().reset_index(name='counts')
print(train_counts)

Number of unique Member_number entries: 22401
         uid  counts
0          0       5
1          1       5
2          2       5
3          3       5
4          4       5
...      ...     ...
22396  37098       1
22397  37101       1
22398  37129       2
22399  37135       1
22400  37219       1

[22401 rows x 2 columns]


In [29]:
# Check data - Test set
# Count the number of customer
unique_members = test_df['uid'].nunique()
print("Number of unique Member_number entries:", unique_members)

# Count the number of basket for each customer
test_counts = test_df.groupby('uid').size().reset_index(name='counts')
print(test_counts)

Number of unique Member_number entries: 22401
         uid  counts
0          0       5
1          1       5
2          2       5
3          3       5
4          4       5
...      ...     ...
22396  37098       1
22397  37101       1
22398  37129       2
22399  37135       1
22400  37219       1

[22401 rows x 2 columns]


In [30]:
# Assign basket numbers
train_df['basket_number'] = range(len(train_df))
test_df['basket_number'] = range(len(test_df))

print("Training Data:")
print(train_df)
print("\nTesting Data:")
print(test_df)

Training Data:
         uid    TX_DATE     2     3     4     5    6    7    8    9  ... 22  \
0          0 2019-03-03     0    24    24    26   27   28   46   47  ...      
1          0 2019-03-24    28    28   583   583   99   99   24   24  ...      
2          0 2019-04-07    28    24   587  2420                      ...      
3          0 2019-04-14   152  1163  1163  1163  397  397  152  152  ...      
4          0 2019-04-24    35    35    35   328  955  955  955       ...      
...      ...        ...   ...   ...   ...   ...  ...  ...  ...  ...  ... ..   
62204  37101 2020-01-17  2136                                        ...      
62205  37129 2020-02-20    17   100                                  ...      
62206  37129 2019-03-29    17  3543                                  ...      
62207  37135 2019-04-19  2012                                        ...      
62208  37219 2019-10-01   899                                        ...      

      23 24 25 26 27 28 29 30 basket

### 7. Generate 3 Files as Input for the Model:

In [ ]:
import datetime

# Training set - train_b2i
# Drop the original 'uid' column 
train_b2i = train_df.drop(columns=['uid'])

# Assign 'basket_number' as the first column
columns = ['basket_number'] + [col for col in train_b2i.columns if col != 'basket_number']
train_b2i = train_b2i[columns]

# Replace NaN with empty strings
train_b2i = train_b2i.fillna('')

# Convert all values to strings, handling empty values and non-string types
def clean_and_convert(value):
    if pd.isna(value) or value == '':
        return ''
    try:
        if isinstance(value, (str, int)):
            return str(int(value))
        elif isinstance(value, float) and not pd.isna(value):
            return str(int(value))
        else:
            return ''
    except ValueError:
        return ''

for col in train_b2i.columns[1:]:
    train_b2i[col] = train_b2i[col].apply(clean_and_convert)

# Formatting data
def format_row(row):
    return ' '.join(str(x) for x in row if x != '')

# Add timestamps
train_b2i['timestamp'] = train_df['TX_DATE'].dt.strftime('%Y%m%d')

formatted_rows = train_b2i.apply(lambda row: ' '.join(str(x).strip() for x in row if x != ''), axis=1)

# formatted_rows = train_b2i.apply(format_row, axis=1)
def validate_timestamp(row):
    try:

        return row
    except ValueError:
        return None

formatted_rows = formatted_rows.apply(validate_timestamp).dropna()  


# Export the data as a text file without column names
with open(f'{path_name}/train_b2i.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}/train_b2i.txt')

Data exported to D:\Jupyter\4742\Summer2024\FinalSubmission\Code\DataPreparationandTransformation\multimodal_data\data\multimodalwithres/train_b2i.txt


In [ ]:
# Test set - test_b2i
# Drop the original 'uid' column 
test_b2i = test_df.drop(columns=['uid'])

# Assign 'basket_number' as the first column
columns = ['basket_number'] + [col for col in test_b2i.columns if col != 'basket_number']
test_b2i = test_b2i[columns]

# Replace NaN with empty strings
test_b2i = test_b2i.fillna('')

# Convert all values to strings, handling empty values and non-string types
for col in test_b2i.columns[1:]:  
    test_b2i[col] = test_b2i[col].apply(clean_and_convert)

# Formatting function to join values with space, ignoring empty strings
def format_row(row):
    return ' '.join(str(x) for x in row if x != '')

# Add timestamps
test_b2i['timestamp'] = test_df['TX_DATE'].dt.strftime('%Y%m%d')

formatted_rows = test_b2i.apply(lambda row: ' '.join(str(x).strip() for x in row if x != ''), axis=1)
formatted_rows = formatted_rows.apply(validate_timestamp).dropna()  # 移除无效行

# Formatting data
# formatted_rows = test_b2i.apply(format_row, axis=1)

# Export the data as a text file without column names
with open(f'{path_name}/test_b2i.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}/test_b2i.txt')

Data exported to D:\Jupyter\4742\Summer2024\FinalSubmission\Code\DataPreparationandTransformation\multimodal_data\data\multimodalwithres/test_b2i.txt


In [33]:
# train_u2b
# Create a basket for each user.
grouped_df = train_df.groupby('uid')['basket_number'].apply(list).reset_index()
expanded_df = grouped_df['basket_number'].apply(pd.Series)
expanded_df.columns = [f'basket_{i+1}' for i in expanded_df.columns]

# Concatenate the expanded columns with the original 'uid' column
train_u2b = pd.concat([grouped_df['uid'], expanded_df], axis=1)

# Replace NaN with empty strings
train_u2b = train_u2b.fillna('')

# Assign 'uid' as the first column
columns = ['uid'] + [col for col in train_u2b.columns if col != 'uid']
train_u2b = train_u2b[columns]

# Convert all values to strings, handling empty values and non-string types
for col in train_u2b.columns[1:]:
    train_u2b[col] = train_u2b[col].apply(clean_and_convert)

# Formatting data
formatted_rows = train_u2b.apply(format_row, axis=1)

# Export the data as a text file without column names
with open(f'{path_name}/train_u2b.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}/train_u2b.txt')

Data exported to D:\Jupyter\4742\Summer2024\FinalSubmission\Code\DataPreparationandTransformation\multimodal_data\data\multimodalwithres/train_u2b.txt


In [34]:
# test_u2b
grouped_df_test = test_df.groupby('uid')['basket_number'].apply(list).reset_index()
expanded_df_test = grouped_df_test['basket_number'].apply(pd.Series)
expanded_df_test.columns = [f'basket_{i+1}' for i in expanded_df_test.columns]

test_u2b = pd.concat([grouped_df_test['uid'], expanded_df_test], axis=1)
test_u2b = test_u2b.fillna('')

columns = ['uid'] + [col for col in test_u2b.columns if col != 'uid']
test_u2b = test_u2b[columns]

for col in test_u2b.columns[1:]:
    test_u2b[col] = test_u2b[col].apply(clean_and_convert)

formatted_rows_test = test_u2b.apply(format_row, axis=1)

with open(f'{path_name}/test_u2b.txt', 'w') as file:
    for row in formatted_rows_test:
        file.write(row + '\n')